# CluSTER in miniature: cluster by learning effect, sample balanced, shard balanced

Companion to the 2026-11-03 post on KAIST's CluSTER (EMNLP 2026, arXiv 2609.12584). Headline numbers are KAIST's; the clustering/GPU demos below are small toys that reproduce the *mechanism*, not the paper's models.

In [ ]:
# 1. Headline arithmetic: 69.6% less time, ~31% of the data
full_hours, kept_frac, reduction = 10.0, 0.31, 0.696
cluster_hours = full_hours * (1 - reduction)
speedup = 1 / (1 - reduction)
print(f"CluSTER time: {cluster_hours:.2f} h (vs 10 h full)")
print(f"Speedup: {speedup:.2f}x ; data kept: {kept_frac*100:.0f}% ; data dropped: {(1-kept_frac)*100:.0f}%")
assert abs(cluster_hours - 3.04) < 0.01
assert abs(speedup - 3.29) < 0.01
assert kept_frac + (1 - kept_frac) == 1.0

## 2. The core idea: two examples can look different and teach the same thing

CluSTER scores each example by its *effect on learning* (its influence), clusters examples with similar effects, then samples across clusters. Topic clustering would split 'fix a loop bug in Python' and 'fix a loop bug in Java' apart; influence clustering can put them together, because both teach the same debugging skill.

In [ ]:
# 2. Toy: 200 examples, 5 skill clusters with skewed sizes (like real data)
import random
random.seed(7)
sizes = {'loops': 90, 'sql-joins': 50, 'regex': 30, 'recursion': 20, 'pointers': 10}
examples = [(skill, i) for skill, n in sizes.items() for i in range(n)]
assert len(examples) == 200
# Balanced sample: 31% of 200 = 62, spread proportionally but capped so no skill dominates
# (stratified: take ~31% from EACH cluster, so every skill survives)
balanced = []
for skill, n in sizes.items():
    k = max(1, round(n * 0.31))
    balanced += [(skill, i) for i in range(k)]
print('balanced sample size:', len(balanced))
print('skills covered by balanced sample:', sorted({s for s, _ in balanced}))
assert len(balanced) == 62 or abs(len(balanced) - 62) <= 5
assert sorted({s for s, _ in balanced}) == sorted(sizes)  # all 5 skills survive

## 3. Balanced allocation across GPUs

Random sharding can hand one GPU a batch of near-duplicates while another gets the variety. Round-robin dealing by cluster evens both the count and the mix.

In [ ]:
# 3. Toy: deal 100 examples across 4 GPUs, cluster by cluster (round-robin)
clusters = {f'skill-{c}': 25 for c in range(4)}
gpus = [[] for _ in range(4)]
for skill in clusters:
    for i in range(25):
        gpus[(i + list(clusters).index(skill)) % 4].append((skill, i))
counts = [len(g) for g in gpus]
print('examples per GPU:', counts)
assert counts == [25, 25, 25, 25]
assert all(len({s for s, _ in g}) == 4 for g in gpus)  # every GPU sees every skill

## 4. What this does NOT prove

The toy above shows the mechanism, not the result. KAIST's 69.6% is a best case on 7B/13B models, three coding+medical datasets, eight benchmarks, mostly 4 GPUs. It does not mean 31% of *your* data suffices, and the influence-scoring step itself costs compute that the headline time saving must repay. Always re-measure on your own workload before cutting data.

In [ ]:
# 4. Sanity: the saving must exceed the selection cost
# If scoring+clustering costs C hours, net saving = 6.96 - C. Break-even is C = 6.96 h.
for selection_cost in [0.5, 2.0, 6.96, 8.0]:
    net = 6.96 - selection_cost
    print(f"selection cost {selection_cost} h -> net saving {net:.2f} h {'(still ahead)' if net > 0 else '(not worth it)'}")
assert 6.96 - 0.5 > 0